# ARES Terrain Segmentation Baseline v0.1 — Kaggle Edition

This notebook trains a lightweight PyTorch U-Net on a bounded, reproducible sample from the `hassanjbara/AI4MARS` Hugging Face dataset mirror. The experiment uses the `train` split, 640 selected labeled samples, and a preliminary image-level 512/128 train/validation split. It is a baseline experiment, not a claim of full-dataset AI4Mars performance.


## Scope and safeguards

- Uses the AI4Mars MSL v0.1 Hugging Face mirror (`hassanjbara/AI4MARS`); equivalence to the official AI4Mars v0.6 release is not established.
- Inspects at most 2,000 streamed records and collects up to 640 usable labeled samples.
- Uses four semantic classes: `0=soil`, `1=bedrock`, `2=sand`, `3=big rock`; `255=ignore/unlabeled`.
- The 512/128 split is a preliminary image-level split. Rover-sequence grouping and duplicate leakage have not been verified.
- Ignore pixels (`255`) are excluded from the loss and evaluation metrics.
- Dataset tensors, checkpoints, and results are stored under `/kaggle/working/ARES`, not in Git. Download important outputs before the Kaggle session ends.


## 1. Runtime and dependency verification


In [ ]:
import importlib.util
import platform
import sys

REQUIRED_PACKAGES = ["torch", "numpy", "PIL", "matplotlib", "datasets"]
missing_dependencies = [
    package for package in REQUIRED_PACKAGES
    if importlib.util.find_spec(package) is None
]

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Missing dependencies:", missing_dependencies)

if missing_dependencies:
    raise RuntimeError(
        f"Missing required packages: {missing_dependencies}. "
        "This notebook does not install packages automatically."
    )

import torch
import numpy as np
import PIL
import matplotlib
import datasets

print("PyTorch:", torch.__version__)
print("Datasets:", datasets.__version__)
print("NumPy:", np.__version__)
print("Pillow:", PIL.__version__)
print("Matplotlib:", matplotlib.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for this Kaggle training run.")

GPU_NAME = torch.cuda.get_device_name(0)
print("Selected GPU:", GPU_NAME)

if "T4" not in GPU_NAME:
    raise RuntimeError(f"Expected a Tesla T4 GPU; found {GPU_NAME}.")

DEVICE = torch.device("cuda:0")
print("Training device:", DEVICE)


## 2. Reproducibility and experiment configuration


In [ ]:
import os
import json
import random
from pathlib import Path

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DATASET_ID = "hassanjbara/AI4MARS"
DATASET_SPLIT = "train"
MAX_RAW_RECORDS = 2_000
TARGET_SAMPLES = 640
TRAIN_SAMPLES = 512
VALIDATION_SAMPLES = 128
IMAGE_SIZE = (256, 256)
NUM_CLASSES = 4
IGNORE_INDEX = 255
CLASS_NAMES = ["Soil", "Bedrock", "Sand", "Big Rock"]

BATCH_SIZE = 8
EPOCHS = 5
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.0
RESUME_CHECKPOINT = False

NUM_WORKERS = 2
PIN_MEMORY = True

PROJECT_DIR = Path("/kaggle/working/ARES")
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
RESULTS_DIR = PROJECT_DIR / "results"
CHECKPOINT_PATH = CHECKPOINT_DIR / "terrain_unet_best.pt"
AUDIT_PATH = RESULTS_DIR / "ares_data_audit.json"
METRICS_PATH = RESULTS_DIR / "validation_metrics.json"
HISTORY_PATH = RESULTS_DIR / "training_history.json"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

assert TRAIN_SAMPLES + VALIDATION_SAMPLES == TARGET_SAMPLES
assert NUM_CLASSES == len(CLASS_NAMES)
assert IGNORE_INDEX == 255
assert IMAGE_SIZE == (256, 256)
assert BATCH_SIZE > 0 and EPOCHS > 0 and LEARNING_RATE > 0

CONFIG = {
    "experiment": "ARES Terrain Segmentation Baseline v0.1",
    "seed": SEED,
    "dataset_id": DATASET_ID,
    "dataset_split": DATASET_SPLIT,
    "max_raw_records": MAX_RAW_RECORDS,
    "target_samples": TARGET_SAMPLES,
    "train_samples": TRAIN_SAMPLES,
    "validation_samples": VALIDATION_SAMPLES,
    "image_size": IMAGE_SIZE,
    "num_classes": NUM_CLASSES,
    "class_names": CLASS_NAMES,
    "ignore_index": IGNORE_INDEX,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "resume_checkpoint": RESUME_CHECKPOINT,
    "device": str(DEVICE),
    "gpu": GPU_NAME,
    "num_workers": NUM_WORKERS,
    "pin_memory": PIN_MEMORY,
    "checkpoint_path": str(CHECKPOINT_PATH),
    "results_dir": str(RESULTS_DIR),
}

print("ARES experiment configuration")
print(json.dumps(CONFIG, indent=2))


## 3. Bounded streaming dataset collection

The dataset is streamed with a fixed shuffle seed and bounded buffer. The collector stops after 2,000 raw records or after 640 usable labeled samples have been collected, whichever happens first. Images are converted to grayscale and resized with bilinear interpolation; masks use nearest-neighbor interpolation to preserve class IDs.


In [ ]:
import io
import numpy as np
import torch

from PIL import Image, UnidentifiedImageError
from datasets import load_dataset, Image as HFImage


# ============================================================
# 1. EXPLICIT IMAGE DECODING
# ============================================================

def decode_image(value):

    if value is None:
        raise ValueError("Image value is None.")

    if isinstance(value, Image.Image):
        return value.copy()

    if isinstance(value, dict):

        if value.get("bytes") is not None:
            value = value["bytes"]

        elif value.get("path") is not None:
            value = value["path"]

        else:
            raise ValueError("Invalid image mapping.")

    if isinstance(value, (bytes, bytearray, memoryview)):
        image = Image.open(io.BytesIO(bytes(value)))

    elif isinstance(value, str):
        image = Image.open(value)

    else:
        raise TypeError(f"Unsupported image type: {type(value)}")

    image.load()
    return image.copy()


# ============================================================
# 2. IMAGE AND MASK PREPROCESSING
# ============================================================

def prepare_pair(image, mask):

    if image.size != mask.size:
        raise ValueError(
            f"Image-mask dimension mismatch: "
            f"{image.size} vs {mask.size}"
        )

    image = image.convert("L").resize(
        IMAGE_SIZE[::-1],
        Image.Resampling.BILINEAR
    )

    mask = mask.convert("L").resize(
        IMAGE_SIZE[::-1],
        Image.Resampling.NEAREST
    )

    image_array = np.asarray(image, dtype=np.float32) / 255.0
    mask_array = np.asarray(mask, dtype=np.uint8)

    image_tensor = torch.from_numpy(
        image_array.copy()
    ).unsqueeze(0)

    mask_tensor = torch.from_numpy(
        mask_array.astype(np.int64)
    )

    return image_tensor, mask_tensor


# ============================================================
# 3. LOAD STREAMING DATASET
# ============================================================

stream = load_dataset(
    DATASET_ID,
    split=DATASET_SPLIT,
    streaming=True
)

stream = stream.cast_column(
    "image",
    HFImage(decode=False)
)

stream = stream.cast_column(
    "label_mask",
    HFImage(decode=False)
)

stream = stream.shuffle(
    seed=SEED,
    buffer_size=MAX_RAW_RECORDS
)


# ============================================================
# 4. VERIFY DECODING CONFIGURATION
# ============================================================

image_feature = stream.features["image"]
mask_feature = stream.features["label_mask"]

assert image_feature.decode is False
assert mask_feature.decode is False

print("Image decoding:", image_feature.decode)
print("Mask decoding:", mask_feature.decode)
print("Feature decoding checks passed.")


# ============================================================
# 5. INITIALIZE COLLECTION
# ============================================================

selected_images = []
selected_masks = []
selected_records = []

raw_records_inspected = 0

skipped_missing_pair = 0
skipped_without_labels = 0
skipped_all_null = 0
skipped_corrupt_image = 0
skipped_corrupt_mask = 0
skipped_dimension_mismatch = 0

ALLOWED_IDS = {
    0, 1, 2, 3, IGNORE_INDEX
}


# ============================================================
# 6. STREAM AND COLLECT
# ============================================================

for raw_index, record in enumerate(stream):

    if raw_records_inspected >= MAX_RAW_RECORDS:
        break

    if len(selected_images) >= TARGET_SAMPLES:
        break

    raw_records_inspected += 1

    image_raw = record.get("image")
    mask_raw = record.get("label_mask")

    if image_raw is None or mask_raw is None:
        skipped_missing_pair += 1
        continue

    if record.get("has_labels") is not True:
        skipped_without_labels += 1
        continue

    try:
        image = decode_image(image_raw)

    except (
        UnidentifiedImageError,
        OSError,
        ValueError,
        TypeError
    ):
        skipped_corrupt_image += 1
        continue

    try:
        mask = decode_image(mask_raw)

    except (
        UnidentifiedImageError,
        OSError,
        ValueError,
        TypeError
    ):
        skipped_corrupt_mask += 1
        continue

    if image.size != mask.size:
        skipped_dimension_mismatch += 1
        continue

    mask = mask.convert("L")
    mask_array = np.asarray(mask)

    observed_ids = set(
        np.unique(mask_array).tolist()
    )

    unexpected_ids = observed_ids - ALLOWED_IDS

    if unexpected_ids:
        raise ValueError(
            f"Unexpected mask IDs at record {raw_index}: "
            f"{sorted(unexpected_ids)}"
        )

    if not np.isin(
        mask_array, [0, 1, 2, 3]
    ).any():
        skipped_all_null += 1
        continue

    image_tensor, mask_tensor = prepare_pair(
        image, mask
    )

    selected_images.append(image_tensor)
    selected_masks.append(mask_tensor)

    selected_records.append({
        "raw_index": raw_index,
        "image_size": image.size,
        "mask_ids": sorted(observed_ids)
    })


# ============================================================
# 7. COLLECTION VALIDATION
# ============================================================

usable_samples = len(selected_images)

if usable_samples != TARGET_SAMPLES:
    raise RuntimeError(
        f"Expected {TARGET_SAMPLES} samples, "
        f"collected {usable_samples}.\n"
        f"Raw records: {raw_records_inspected}\n"
        f"Missing pairs: {skipped_missing_pair}\n"
        f"Without labels: {skipped_without_labels}\n"
        f"All-null: {skipped_all_null}\n"
        f"Corrupt images: {skipped_corrupt_image}\n"
        f"Corrupt masks: {skipped_corrupt_mask}\n"
        f"Dimension mismatch: {skipped_dimension_mismatch}"
    )


# ============================================================
# 8. CREATE CPU TENSORS
# ============================================================

selected_images = torch.stack(selected_images).contiguous()
selected_masks = torch.stack(selected_masks).contiguous()

assert selected_images.device.type == "cpu"
assert selected_masks.device.type == "cpu"

assert torch.isfinite(selected_images).all()

assert selected_images.min() >= 0
assert selected_images.max() <= 1

assert set(torch.unique(selected_masks).tolist()) <= ALLOWED_IDS

assert selected_images.shape == (
    TARGET_SAMPLES, 1, IMAGE_SIZE[0], IMAGE_SIZE[1]
)

assert selected_masks.shape == (
    TARGET_SAMPLES, IMAGE_SIZE[0], IMAGE_SIZE[1]
)


# ============================================================
# 9. COLLECTION REPORT
# ============================================================

collection_report = {
    "dataset": DATASET_ID,
    "split": DATASET_SPLIT,
    "mirror_version": "0.1 (verify against dataset metadata)",
    "raw_records_inspected": raw_records_inspected,
    "usable_samples": usable_samples,
    "skipped_missing_pair": skipped_missing_pair,
    "skipped_without_labels": skipped_without_labels,
    "skipped_all_null": skipped_all_null,
    "skipped_corrupt_image": skipped_corrupt_image,
    "skipped_corrupt_mask": skipped_corrupt_mask,
    "skipped_dimension_mismatch": skipped_dimension_mismatch,
    "image_tensor_shape": tuple(selected_images.shape),
    "mask_tensor_shape": tuple(selected_masks.shape),
    "storage_device": str(selected_images.device)
}

print("\n========== ARES DATASET COLLECTION REPORT ==========")

for key, value in collection_report.items():
    print(f"{key}: {value}")

print("\nDataset collection completed successfully.")


### Dataset pixel distribution


In [ ]:
class_names_with_ignore = {
    0: "Soil",
    1: "Bedrock",
    2: "Sand",
    3: "Big Rock",
    IGNORE_INDEX: "Ignored",
}

all_mask_values = selected_masks.reshape(-1)
total_pixels = all_mask_values.numel()
print("TERRAIN CLASS DISTRIBUTION")
print("=" * 48)
for class_id, class_name in class_names_with_ignore.items():
    count = int((all_mask_values == class_id).sum().item())
    percentage = 100.0 * count / total_pixels
    print(f"{class_name:12s} | Pixels: {count:12,d} | Percentage: {percentage:7.3f}%")


### Reproducible train/validation split

This is a seeded random image-level split. It prevents a sample from appearing in both subsets, but it does not establish rover-sequence independence.


In [ ]:
generator = torch.Generator().manual_seed(SEED)
indices = torch.randperm(TARGET_SAMPLES, generator=generator)
train_indices = indices[:TRAIN_SAMPLES]
validation_indices = indices[TRAIN_SAMPLES:]

train_images = selected_images[train_indices]
train_masks = selected_masks[train_indices]
validation_images = selected_images[validation_indices]
validation_masks = selected_masks[validation_indices]

assert train_images.shape[0] == TRAIN_SAMPLES
assert train_masks.shape[0] == TRAIN_SAMPLES
assert validation_images.shape[0] == VALIDATION_SAMPLES
assert validation_masks.shape[0] == VALIDATION_SAMPLES
assert set(train_indices.tolist()).isdisjoint(validation_indices.tolist())

print("TRAIN/VALIDATION SPLIT SUCCESSFUL")
print("Training images:", tuple(train_images.shape))
print("Training masks:", tuple(train_masks.shape))
print("Validation images:", tuple(validation_images.shape))
print("Validation masks:", tuple(validation_masks.shape))
print("Overlapping indices: 0")


### Compact data audit


In [ ]:
def calculate_class_pixel_counts(masks):
    counts = torch.bincount(
        masks.reshape(-1).long(),
        minlength=IGNORE_INDEX + 1,
    )
    return {str(class_id): int(counts[class_id].item()) for class_id in range(NUM_CLASSES)}

def calculate_ignored_pixel_count(masks):
    return int((masks == IGNORE_INDEX).sum().item())

audit_report = {
    "experiment": "ARES Terrain Segmentation Baseline v0.1",
    "dataset": DATASET_ID,
    "split": DATASET_SPLIT,
    "raw_records_inspected": int(raw_records_inspected),
    "usable_samples": int(selected_images.shape[0]),
    "skipped_records": {
        "missing_image_or_mask": int(skipped_missing_pair),
        "without_labels": int(skipped_without_labels),
        "all_null_mask": int(skipped_all_null),
        "corrupt_images": int(skipped_corrupt_image),
        "corrupt_masks": int(skipped_corrupt_mask),
        "dimension_mismatch": int(skipped_dimension_mismatch),
    },
    "class_distribution_pixels": {
        "all_samples": calculate_class_pixel_counts(selected_masks),
        "training": calculate_class_pixel_counts(train_masks),
        "validation": calculate_class_pixel_counts(validation_masks),
    },
    "ignored_pixels": {
        "all_samples": calculate_ignored_pixel_count(selected_masks),
        "training": calculate_ignored_pixel_count(train_masks),
        "validation": calculate_ignored_pixel_count(validation_masks),
    },
    "train_samples": int(train_images.shape[0]),
    "validation_samples": int(validation_images.shape[0]),
    "image_shape": list(selected_images.shape),
    "mask_shape": list(selected_masks.shape),
    "tensor_storage_device": str(selected_images.device),
    "training_device": str(DEVICE),
}
AUDIT_PATH.write_text(json.dumps(audit_report, indent=2) + "\n", encoding="utf-8")
print(json.dumps(audit_report, indent=2))
print("Audit saved to:", AUDIT_PATH)


### PyTorch DataLoaders


In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(train_images, train_masks)
validation_dataset = TensorDataset(validation_images, validation_masks)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

images, masks = next(iter(train_loader))
assert images.shape == (BATCH_SIZE, 1, *IMAGE_SIZE)
assert masks.shape == (BATCH_SIZE, *IMAGE_SIZE)
assert images.dtype == torch.float32
assert masks.dtype == torch.int64

print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))
print("Image batch:", tuple(images.shape), images.dtype, images.device)
print("Mask batch:", tuple(masks.shape), masks.dtype, masks.device)


## 4. Training-only class weights

Class weights are calculated from training pixels only. Ignore pixels (`255`) are excluded. Median-frequency balancing is used to reduce the effect of severe class imbalance without using raw inverse-frequency weights.


In [ ]:
class_counts = torch.bincount(
    train_masks[train_masks != IGNORE_INDEX].long(),
    minlength=NUM_CLASSES,
).float()

if torch.any(class_counts <= 0):
    raise RuntimeError(f"At least one training class has zero pixels: {class_counts.tolist()}")

frequencies = class_counts / class_counts.sum()
median_frequency = frequencies.median()
class_weights = median_frequency / frequencies
class_weights = class_weights / class_weights.mean()
class_weights = class_weights.to(DEVICE)

print("TRAINING CLASS WEIGHTS")
for index, name in enumerate(CLASS_NAMES):
    print(
        f"{name:12s} | Pixels: {int(class_counts[index].item()):,} | "
        f"Frequency: {frequencies[index].item() * 100:.3f}% | "
        f"Weight: {class_weights[index].item():.4f}"
    )
print("Ignored pixel ID:", IGNORE_INDEX)


## 5. Tiny U-Net model


In [ ]:
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class TinyUNet(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.enc1 = DoubleConv(1, 16)
        self.enc2 = DoubleConv(16, 32)
        self.enc3 = DoubleConv(32, 64)
        self.bottleneck = DoubleConv(64, 128)
        self.pool = nn.MaxPool2d(2)
        self.up3 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec3 = DoubleConv(128, 64)
        self.up2 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec2 = DoubleConv(64, 32)
        self.up1 = nn.ConvTranspose2d(32, 16, 2, stride=2)
        self.dec1 = DoubleConv(32, 16)
        self.head = nn.Conv2d(16, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        bottleneck = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(bottleneck), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.head(d1)

# Fresh initialization: the real training run starts from these weights.
model = TinyUNet().to(DEVICE)
parameter_count = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
print("Trainable parameters:", parameter_count)
assert parameter_count == 482500


### Forward-pass and loss setup


In [ ]:
criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    ignore_index=IGNORE_INDEX,
)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

model.eval()
with torch.no_grad():
    test_images, test_masks = next(iter(train_loader))
    test_logits = model(test_images.to(DEVICE, non_blocking=True))

expected_output_shape = (BATCH_SIZE, NUM_CLASSES, *IMAGE_SIZE)
assert test_logits.shape == expected_output_shape
assert torch.isfinite(test_logits).all()
print("Forward pass successful")
print("Input:", tuple(test_images.shape))
print("Output:", tuple(test_logits.shape))
print("Target:", tuple(test_masks.shape))
print("Device:", next(model.parameters()).device)
print("Loss:", criterion)
print("Optimizer:", optimizer.__class__.__name__)


## 6. Kaggle checkpoint directory

Checkpoint files are written to Kaggle working storage, outside the Git repository. Kaggle working storage is temporary; download the best checkpoint and result JSON files before the session ends.


In [ ]:
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Checkpoint directory ready:", CHECKPOINT_DIR)
print("Best checkpoint path:", CHECKPOINT_PATH)
print("Results directory:", RESULTS_DIR)


## 7. Training loop

The baseline runs for five epochs. It records mean loss per valid batch and saves the checkpoint with the lowest validation loss. Set `RESUME_CHECKPOINT=True` only when intentionally resuming from a compatible checkpoint.


In [ ]:
import time

history = {"train_loss": [], "validation_loss": [], "epoch_seconds": []}
start_epoch = 0
best_validation_loss = float("inf")

if RESUME_CHECKPOINT and CHECKPOINT_PATH.exists():
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    start_epoch = int(checkpoint["epoch"]) + 1
    best_validation_loss = float(checkpoint["best_validation_loss"])
    history = checkpoint.get("history", history)
    print("Resumed after epoch:", start_epoch)

if start_epoch >= EPOCHS:
    raise RuntimeError(
        f"Checkpoint already reached epoch {start_epoch}; increase EPOCHS or disable resume."
    )

for epoch in range(start_epoch, EPOCHS):
    epoch_start = time.time()
    model.train()
    train_loss_sum = 0.0
    train_batches = 0

    for batch_images, batch_masks in train_loader:
        if (batch_masks != IGNORE_INDEX).sum().item() == 0:
            continue
        batch_images = batch_images.to(DEVICE, non_blocking=True)
        batch_masks = batch_masks.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(batch_images)
        loss = criterion(logits, batch_masks)
        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite training loss in epoch {epoch + 1}.")
        loss.backward()
        optimizer.step()
        train_loss_sum += float(loss.item())
        train_batches += 1

    model.eval()
    validation_loss_sum = 0.0
    validation_batches = 0
    with torch.no_grad():
        for batch_images, batch_masks in validation_loader:
            if (batch_masks != IGNORE_INDEX).sum().item() == 0:
                continue
            batch_images = batch_images.to(DEVICE, non_blocking=True)
            batch_masks = batch_masks.to(DEVICE, non_blocking=True)
            validation_loss = criterion(model(batch_images), batch_masks)
            if not torch.isfinite(validation_loss):
                raise RuntimeError(f"Non-finite validation loss in epoch {epoch + 1}.")
            validation_loss_sum += float(validation_loss.item())
            validation_batches += 1

    if train_batches == 0 or validation_batches == 0:
        raise RuntimeError("An epoch had no valid supervised batches.")

    train_loss = train_loss_sum / train_batches
    validation_loss = validation_loss_sum / validation_batches
    epoch_seconds = time.time() - epoch_start
    history["train_loss"].append(train_loss)
    history["validation_loss"].append(validation_loss)
    history["epoch_seconds"].append(epoch_seconds)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"validation_loss={validation_loss:.4f} | "
        f"time={epoch_seconds:.1f}s"
    )

    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_validation_loss": best_validation_loss,
                "history": history,
                "configuration": CONFIG,
            },
            CHECKPOINT_PATH,
        )
        print("Saved best checkpoint:", CHECKPOINT_PATH)

HISTORY_PATH.write_text(json.dumps(history, indent=2) + "\n", encoding="utf-8")
print("Training complete. Best validation loss:", best_validation_loss)
print("Training history saved:", HISTORY_PATH)


## 8. Best-checkpoint validation metrics

The best validation-loss checkpoint is reloaded before metric calculation. Metrics describe only this preliminary 128-image validation subset; they are not full-dataset or independent expert-test results.


In [ ]:
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Best checkpoint not found at {CHECKPOINT_PATH}. Run the training cell first."
    )

best_checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(best_checkpoint["model_state_dict"])
print("Loaded best checkpoint from epoch:", int(best_checkpoint["epoch"]) + 1)

def update_confusion_matrix(confusion, predictions, targets):
    valid = targets != IGNORE_INDEX
    valid_targets = targets[valid].to(torch.int64)
    valid_predictions = predictions[valid].to(torch.int64)
    if valid_targets.numel() == 0:
        return
    encoded = valid_targets * NUM_CLASSES + valid_predictions
    batch_confusion = torch.bincount(
        encoded, minlength=NUM_CLASSES * NUM_CLASSES
    ).reshape(NUM_CLASSES, NUM_CLASSES)
    confusion += batch_confusion.cpu()

def evaluate(model, loader):
    model.eval()
    confusion = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.int64)
    loss_sum = 0.0
    valid_batches = 0
    with torch.no_grad():
        for batch_images, batch_masks in loader:
            if (batch_masks != IGNORE_INDEX).sum().item() == 0:
                continue
            batch_images = batch_images.to(DEVICE, non_blocking=True)
            batch_masks_device = batch_masks.to(DEVICE, non_blocking=True)
            logits = model(batch_images)
            loss_sum += float(criterion(logits, batch_masks_device).item())
            predictions = logits.argmax(dim=1).cpu()
            update_confusion_matrix(confusion, predictions, batch_masks)
            valid_batches += 1

    if valid_batches == 0 or confusion.sum().item() == 0:
        raise RuntimeError("Validation contains no valid labeled pixels.")

    true_positive = torch.diag(confusion).float()
    ground_truth_count = confusion.sum(dim=1).float()
    predicted_count = confusion.sum(dim=0).float()
    union = ground_truth_count + predicted_count - true_positive
    per_class_iou = torch.full((NUM_CLASSES,), float("nan"))
    classes_present_in_ground_truth = ground_truth_count > 0
    per_class_iou[classes_present_in_ground_truth] = (
        true_positive[classes_present_in_ground_truth]
        / union[classes_present_in_ground_truth].clamp_min(1.0)
    )
    pixel_accuracy = true_positive.sum() / confusion.sum().float()

    return {
        "validation_loss": loss_sum / valid_batches,
        "pixel_accuracy": float(pixel_accuracy.item()),
        "per_class_iou": {
            CLASS_NAMES[index]: (
                None if torch.isnan(per_class_iou[index])
                else float(per_class_iou[index].item())
            )
            for index in range(NUM_CLASSES)
        },
        "mean_iou_present_ground_truth_classes": float(torch.nanmean(per_class_iou).item()),
        "confusion_matrix": confusion,
    }

metrics = evaluate(model, validation_loader)
metrics_for_json = {key: value for key, value in metrics.items() if key != "confusion_matrix"}
METRICS_PATH.write_text(
    json.dumps(metrics_for_json, indent=2) + "\n", encoding="utf-8"
)
print(json.dumps(metrics_for_json, indent=2))
print("Metrics saved to:", METRICS_PATH)


### Validation confusion matrix


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(metrics["confusion_matrix"].numpy(), cmap="Blues")
ax.set_xlabel("Predicted class")
ax.set_ylabel("Ground-truth class")
ax.set_xticks(range(NUM_CLASSES), CLASS_NAMES, rotation=30, ha="right")
ax.set_yticks(range(NUM_CLASSES), CLASS_NAMES)
for row in range(NUM_CLASSES):
    for column in range(NUM_CLASSES):
        ax.text(
            column, row, int(metrics["confusion_matrix"][row, column]),
            ha="center", va="center",
        )
fig.colorbar(image, ax=ax, label="Pixels")
fig.tight_layout()
plt.show()


### Qualitative predictions


In [ ]:
display_count = min(4, validation_images.shape[0])
model.eval()
with torch.no_grad():
    prediction_batch = model(
        validation_images[:display_count].to(DEVICE, non_blocking=True)
    ).argmax(dim=1).cpu()

fig, axes = plt.subplots(3, display_count, figsize=(4 * display_count, 10), squeeze=False)
for column in range(display_count):
    axes[0, column].imshow(validation_images[column, 0], cmap="gray")
    axes[0, column].set_title("Input")
    axes[1, column].imshow(validation_masks[column], vmin=0, vmax=NUM_CLASSES - 1, cmap="viridis")
    axes[1, column].set_title("Ground truth")
    axes[2, column].imshow(prediction_batch[column], vmin=0, vmax=NUM_CLASSES - 1, cmap="viridis")
    axes[2, column].set_title("Prediction")
    for row in range(3):
        axes[row, column].axis("off")
plt.tight_layout()
plt.show()


## Baseline interpretation and limitations

This experiment trains a small, randomly initialized U-Net on a bounded 640-sample subset. Reported validation loss, pixel accuracy, per-class IoU, and mean IoU apply only to the preliminary image-level validation split. Before drawing stronger conclusions, verify dataset provenance, remove near-duplicate leakage, and create a rover-sequence-aware split and an independent expert-labeled test evaluation. The checkpoint and JSON artifacts are stored under `/kaggle/working/ARES`; download them before the Kaggle session expires.
